#Phase 1: Knowledge ingestion and RAG setup

### Step 1: Install Dependencies & Setup Environment

In this step, we install all the required open-source libraries needed for our local Agentic RAG system:
- **PyPDF**: Extracts text from official NCAIR PDF manuals.
- **LangChain & Community**: Handles text splitting, embeddings, and vector database utilities.
- **Sentence-Transformers**: Runs lightweight vector embedding models locally without APIs.
- **FAISS**: Facebook AI Similarity Search library for fast document retrieval.
- **Gradio**: Builds a clean web interface directly in Colab.

In [ ]:
#Install required system tool (zstd) so the Ollama installer can extract properly
!apt-get update -qq && apt-get install -y -qq zstd

# 1. Install required packages for RAG, vector search, and web UI
!pip install -q pypdf langchain langchain-community sentence-transformers faiss-cpu gradio

# 2. Install and launch Ollama in the background
!curl -fsSL https://ollama.com/install.sh | sh

# Installl  pytesseract pdf2image
!pip install pytesseract pdf2image
!apt-get update -qq && apt-get install -y -qq poppler-utils tesseract-ocr

import subprocess
import time

# Start Ollama server as a background process
subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # Wait for server startup

# 3. Pull Llama 3.2 (3B) model locally
!ollama pull llama3.2:3b

print("\n✅ Step 1 Complete: All dependencies installed and Llama 3.2 model is ready!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver


### Step 2: Automatic PDF Ingestion & FAISS Database

In this step, we scan for and load actual PDF documents:
- **DirectoryLoader & PyPDFLoader**: Scans the Colab root directory and parses all uploaded `.pdf` files.
- **RecursiveCharacterTextSplitter**: Splices the PDF text into 400-character overlapping chunks for fine-grained retrieval.
- **FAISS & Sentence-Transformers**: Encodes the text chunks locally into 384-dimensional vector embeddings and builds the search index.

##convert the PDF pages directly into high-resolution images and run OCR (Optical Character Recognition)

### 1. Install system OCR engine (Tesseract) and PDF render tools (poppler)
!apt-get update -qq && apt-get install -y -qq tesseract-ocr poppler-utils

### 2. Install Python packages for image-based PDF extraction
!pip install -q pytesseract pdf2image

In [ ]:
import os
import pytesseract
from pdf2image import convert_from_path

# Updated LangChain imports for compatibility
try:
    from langchain_core.documents import Document
except ImportError:
    from langchain.schema import Document

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

all_documents = []

# 1. OCR Processing for PDF Manuals
pdf_files = [f for f in os.listdir('.') if f.endswith('.pdf')]
if not pdf_files:
    print("⚠️ No PDF files found in Colab! Skipping PDF OCR processing.")
else:
    pdf_path = pdf_files[0]
    print(f"📄 Processing '{pdf_path}' with OCR...")
    images = convert_from_path(pdf_path)
    for page_num, img in enumerate(images, start=1):
        page_text = pytesseract.image_to_string(img)
        if page_text.strip():
            doc = Document(
                page_content=page_text,
                metadata={"source": pdf_path, "page": page_num}
            )
            all_documents.append(doc)
    print(f"✅ OCR extraction complete for PDF ({len(images)} pages processed).")

# 2. Ingest Text Knowledge Base Files
txt_files = [f for f in os.listdir('.') if f.endswith('.txt')]
if not txt_files:
    print("⚠️ No .txt files found! Make sure 'ncair_knowledge_base.txt' is uploaded.")
else:
    for txt_file in txt_files:
        print(f"📝 Ingesting text knowledge base file: '{txt_file}'...")
        loader = TextLoader(txt_file)
        all_documents.extend(loader.load())

if not all_documents:
    print("⚠️ No document sources were loaded. Please upload your PDF or TXT files and try again.")
else:
    print(f"📚 Total raw documents combined: {len(all_documents)}")

    # 3. Split combined documents into semantic chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=80,
        separators=["\n\n", "\n", " ", ""]
    )
    documents_chunks = text_splitter.split_documents(all_documents)
    print(f"🧩 Created {len(documents_chunks)} document chunks across all sources.")

    # 4. Build FAISS Vector Database with local embeddings
    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    faiss_vector_db = FAISS.from_documents(documents_chunks, embedding_model)

    # 5. Global retriever function for RAG
    def search_ncair_knowledge_base(query: str, top_k: int = 3) -> str:
        """Searches the FAISS vector index for top-k document passages relevant to the user query."""
        docs = faiss_vector_db.similarity_search(query, k=top_k)
        retrieved_passages = []
        for doc in docs:
            source = doc.metadata.get('source', 'Knowledge Base')
            page = doc.metadata.get('page')
            header = f"[{source} - Page {page}]" if page else f"[{source}]"
            retrieved_passages.append(f"{header}\n{doc.page_content.strip()}")
        return "\n---\n".join(retrieved_passages)

    print("\n✅ Step 2 Complete: Multi-Source FAISS Vector Database active!")
    print("\n--- TEST RETRIEVAL OUTPUT FROM COMBINED KNOWLEDGE BASE ---")
    print(search_ncair_knowledge_base("What happens if I pass tests but fail attendance?"))

### Step 3: Agent Tool Engineering (Navigation Router & Guidance Engine)

In this step, we construct custom programmatic tools that extend our chatbot into an active Agent:
- **PORTAL_URL_REGISTRY**: A structured registry mapping key onboarding actions to official NCAIR portal URLs.
- **get_portal_link()**: An agent tool function that generates dynamic, clickable Markdown links for page navigation.
- **get_step_guidance()**: An agent tool function that delivers step-specific troubleshooting instructions for stuck interns.

In [ ]:
# 1. Define centralized URL registry for official NCAIR LMS portal destinations
PORTAL_URL_REGISTRY = {
    # Official portal links from the onboarding manual
    "main": "https://lms.ncair.nitda.gov.ng",
    "login": "https://lms.ncair.nitda.gov.ng/intern/signin",
    "signin": "https://lms.ncair.nitda.gov.ng/intern/signin",
    "ncair_home": "https://ncair.nitda.gov.ng/",

    # Custom/expanded navigation endpoints
    "register": "https://lms.ncair.nitda.gov.ng",
    "profile": "https://lms.ncair.nitda.gov.ng/intern/profile",
    "courses": "https://lms.ncair.nitda.gov.ng/intern/courses",
    "track_selection": "https://lms.ncair.nitda.gov.ng/intern/courses",
    "support": "https://ncair.nitda.gov.ng/contact/"
}

# 2. Agent Tool: Site Router for URL generation
def get_portal_link(action: str) -> str:
    """Returns direct, clickable Markdown hyperlinks for requested portal pages."""
    action_key = action.lower().strip()
    if action_key in PORTAL_URL_REGISTRY:
        url = PORTAL_URL_REGISTRY[action_key]
        clean_title = action_key.replace('_', ' ').title()
        return f"🔗 [Click here to open the {clean_title} Page]({url})"

    # Fallback to main portal link if action key is unrecognized
    return f"🔗 [Click here to visit the NCAIR LMS Landing Page]({PORTAL_URL_REGISTRY['main']})"

# 3. Agent Tool: Step-by-Step Troubleshooting Guidance
def get_step_guidance(step_number: int) -> str:
    """Provides targeted, sequential troubleshooting instructions for specific onboarding stages."""
    guidance_map = {
        1: "📍 **Step 1 (Profile Setup):** Navigate to 'Edit Profile', complete all required bio fields, and save changes.",
        2: "📍 **Step 2 (ID Verification):** Go to 'Documents', upload a valid ID (National ID/Student ID/Passport) under 2MB, and submit.",
        3: "📍 **Step 3 (Track Selection):** Navigate to 'Track Selection' and choose your assigned department (AI, Robotics, Embedded Systems, Data Science).",
        4: "📍 **Step 4 (Final Submission):** Verify all details on your summary dashboard and click 'Submit Final Registration'."
    }
    return guidance_map.get(step_number, "⚠️ Invalid step number. Please specify a step between 1 and 4.")

# 4. Verify tool execution with test calls
print("✅ Step 3 Complete: Agent Navigation & Guidance Tools built successfully!\n")
print("--- TEST TOOL OUTPUT 1 (Registration Link) ---")
print(get_portal_link("register"))

print("\n--- TEST TOOL OUTPUT 2 (Step 2 Guidance) ---")
print(get_step_guidance(2))

### Step 4: Ollama Local LLM & Agent Controller Setup

In this step, we connect our local Llama 3.2 model to the RAG database and agent tools:
- **OLLAMA_API_URL**: Connects locally to `http://localhost:11434/api/generate` without needing external API keys.
- **query_local_llama()**: Formats system prompts and fetches responses from the local Llama 3.2 (3B) instance.
- **agentic_rag_orchestrator()**: The core decision loop that checks query intent, routes to navigation tools or FAISS RAG search, and generates grounded answers.

In [ ]:
import requests
import json

# 1. Define local Ollama API endpoint running inside our Colab background process
OLLAMA_API_URL = "http://localhost:11434/api/generate"

# 2. Local LLM Communication Function
def query_local_llama(prompt: str, model_name: str = "llama3.2:3b") -> str:
    """Sends a prompt directly to the locally running Ollama Llama 3.2 model and returns the response."""
    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False  # Return the complete generated text response as a single JSON payload
    }
    try:
        response = requests.post(OLLAMA_API_URL, json=payload, timeout=120)
        if response.status_code == 200:
            return response.json().get("response", "").strip()
        else:
            return f"⚠️ Error from local LLM engine (Status Code: {response.status_code})"
    except Exception as e:
        return f"⚠️ Could not connect to local Ollama server: {str(e)}"

# 3. Agent Orchestrator & Logic Router
def agentic_rag_orchestrator(user_query: str) -> str:
    """Evaluates user intent and dynamically triggers navigation tools, step guidance, or FAISS vector RAG."""
    query_lower = user_query.lower()

    # Intent 1: User is requesting links or page navigation
    if any(keyword in query_lower for keyword in ["link", "url", "where to", "open", "page", "access", "portal"]):
        if "register" in query_lower or "sign up" in query_lower:
            return f"To begin registration, open the portal page here:\n\n{get_portal_link('register')}"
        elif "log" in query_lower or "sign in" in query_lower:
            return f"To log in to your account, use this direct link:\n\n{get_portal_link('login')}"
        elif "profile" in query_lower:
            return f"You can edit your bio and profile details here:\n\n{get_portal_link('profile')}"
        elif "track" in query_lower:
            return f"Access track selection using this page:\n\n{get_portal_link('track_selection')}"
        elif "support" in query_lower or "help" in query_lower:
            return f"You can reach support directly here:\n\n{get_portal_link('support')}"
        else:
            return f"Here is the link to the main NCAIR LMS landing page:\n\n{get_portal_link('main')}"

    # Intent 2: User indicates they are stuck on a specific onboarding step
    if "stuck" in query_lower or "step" in query_lower:
        if "1" in query_lower or "profile" in query_lower:
            return get_step_guidance(1)
        elif "2" in query_lower or "upload" in query_lower or "id" in query_lower:
            return get_step_guidance(2)
        elif "3" in query_lower or "track" in query_lower:
            return get_step_guidance(3)
        elif "4" in query_lower or "submit" in query_lower:
            return get_step_guidance(4)
        else:
            return "Here is the complete step-by-step onboarding sequence:\n\n" + "\n\n".join([get_step_guidance(i) for i in range(1, 5)])

    # Intent 3: General FAQ or process question -> Route through FAISS Vector RAG + Llama 3.2
    retrieved_context = search_ncair_knowledge_base(user_query, top_k=2)

    # Construct system prompt constraining Llama 3.2 strictly to retrieved RAG context
    system_prompt = f"""
    You are the official NCAIR LMS Onboarding Assistant.
    Answer the intern's question concisely using ONLY the official context provided below.
    If the provided context does not contain enough details to answer, state clearly that they should contact support at support@ncair.nitda.gov.ng.

    --- OFFICIAL NCAIR CONTEXT ---
    {retrieved_context}
    ------------------------------

    User Question: {user_query}
    Answer:
    """

    return query_local_llama(system_prompt)

# 4. Verify agent orchestrator execution
print("✅ Step 4 Complete: Local Ollama LLM & Agent Orchestrator active!\n")
print("--- TEST AGENT EXECUTION 1 (Navigation Intent) ---")
print(agentic_rag_orchestrator("Where can I find the link to register on the portal?"))

print("\n--- TEST AGENT EXECUTION 2 (RAG Search Intent) ---")
print(agentic_rag_orchestrator("What should I do if my document upload fails?"))

✅ Step 4 Complete: Local Ollama LLM & Agent Orchestrator active!

--- TEST AGENT EXECUTION 1 (Navigation Intent) ---
To begin registration, open the portal page here:

🔗 [Click here to open the Register Page](https://lms.ncair.nitda.gov.ng)

--- TEST AGENT EXECUTION 2 (RAG Search Intent) ---
According to the NCAIR Knowledge Base, if your document upload fails, you should check that your uploaded files are in PDF or PNG format and under 2MB in file size.


### Step 5: Web User Interface Deployment using Gradio

In this final phase of our prototype pipeline, we deploy an interactive web interface:
- **chat_response()**: Formats incoming chat messages and forwards them to `agentic_rag_orchestrator()`.
- **gr.ChatInterface**: Builds a modern, responsive chat container with message history rendering.
- **Public URL Sharing**: Setting `share=True` generates a secure external web link to access and test the interface outside Google Colab.

In [ ]:
import gradio as gr

def chat_response(message: str, history) -> str:
    """Receives input from the UI and routes it to the agent orchestrator."""
    return agentic_rag_orchestrator(message)

# Direct ChatInterface initialization without custom HTML/CSS blocks
demo = gr.ChatInterface(
    fn=chat_response,
    title="NCAIR LMS Intelligent Assistant",
    description="Official Virtual Onboarding & Navigation Guide for Interns abd NYSC Members",
    textbox=gr.Textbox(
        placeholder="Ask about registration steps, attendance policies, logbooks, or portal URLs..."
    ),
    chatbot=gr.Chatbot(
        height=480,
        avatar_images=(
            "https://api.iconify.design/lucide:user.svg?color=%23475569",
            "https://api.iconify.design/lucide:bot.svg?color=%2300796b"
        )
    ),
    examples=[
        "Where is the registration link for the portal?",
        "What happens if I pass my tests but fail to meet 75% attendance?",
        "How often do SIWES interns need to sign their logbook?",
        "I am stuck on step 2 of registration",
        "What courses do SIWES interns take in their first cohort?"
    ]
)

print("🚀 Launching Simplified NCAIR Assistant Interface...")

# Launch with teal theme and shareable link
demo.launch(
    theme=gr.themes.Soft(primary_hue="teal", neutral_hue="slate"),
    share=True,
    debug=True
)